# HR Chatbot RAG — Learning Notes

Debugging notes for our RAG pipeline: failures, diagnosis, fixes.

**Project:** `HR_chat_bot`  
**Stack:** LangChain + ChromaDB + Ollama + cross-encoder rerank + Streamlit

**Sections 1–8:** Initial fixes (prompt parroting, retrieval, corpus hygiene)  
**Sections 9–15:** Subscription plans case study, reranking, multi-chunk synthesis, inspection

---

## 1. The Starting Problem

After building the RAG pipeline, a simple HR question produced a useless answer.

### Question

> **Is the vedic marriage platform free?**

### Actual Response (broken)

> Yes, if you don't know the answer based on the provided context, you can simply say that you don't know. The chatbot will then try to provide a comprehensive and logical response using relevant pieces of information from the given context. When answering questions about "how long does it take to register and create a Profile on Vedic Marriage website?", for example, the HR assistant chatbot may say that it takes 10-15 minutes or mention that the form is quite lengthy...

### Sources Retrieved

- `VedicMarriage_Refund_Cancellation_Policy.docx`
- `terms and condition of vedicmarriage@ai.docx`
- `FAQ.docx` (×2)

### Expected Response

> Yes — registration on Vedic Marriage is **free**. Free accounts have limited features; Premium subscription unlocks more connection requests and messaging.

**Goal of this notebook:** Understand *why* the pipeline failed and how each layer was fixed.

---

## 2. Root Cause Overview

RAG can fail at **three independent layers**. All three were involved here:

| Layer | Symptom | Status after fixes |
|-------|---------|-------------------|
| **Retrieval** | Wrong FAQ chunks surfaced; pricing FAQ missed | Fixed (clean corpus + better embeddings) |
| **Generation** | LLM parroted prompt instructions | Fixed (Qwen 2.5 + shorter prompt) |
| **Configuration** | Streamlit ≠ API; random model combos | Fixed (`rag_stacks.py`) |

```
User Question
     │
     ▼
[1] RETRIEVAL  ──►  embed query → search ChromaDB → top-k chunks
     │
     ▼
[2] PROMPT     ──►  instructions + context + question
     │
     ▼
[3] GENERATION ──►  LLM synthesizes answer
     │
     ▼
   Answer + Sources
```

---

## 3. Issue #1 — LLM Parroting the System Prompt

### Symptom

The response repeated prompt instructions almost word-for-word instead of answering from context.

### How We Diagnosed It

1. Compared the output to `qa_template` in `app/services/rag_service.py` — near-verbatim match.
2. Streamlit verbose logs showed the **prompt was correct**; failure happened at **generation**.
3. Model in use: **TinyLlama** (~1.1B parameters) — too weak for long RAG prompts.

### The Old Prompt (too verbose for small models)

In [ ]:
# Old qa_template (simplified excerpt)
OLD_PROMPT = """
You are a helpful HR assistant chatbot. Use the following pieces of context...

If you don't know the answer based on the provided context, just say that you don't know...

When answering:
1. Be comprehensive and detailed
2. Use the context information to provide accurate answers
3. If the context contains multiple relevant pieces, combine them logically
...
"""

### Why TinyLlama Failed

- Started with a hallucinated **"Yes"**
- Then **regurgitated** the numbered instructions from the prompt
- Never synthesized an answer from the retrieved FAQ text

### Fixes Applied

| Fix | Detail |
|-----|--------|
| **Stronger LLM** | Switched to `qwen2.5:1.5b` (~1 GB, fits 8 GB Mac) |
| **Shorter prompt** | 4 lines instead of 17; removed numbered rule list |

### New Prompt (simplified)

```
You are an HR assistant for Vedic Marriage. Answer using only the context below.
Be concise and direct.

If the answer is not in the context, say: "I don't have that information in our HR documents."

Context: {context}
Question: {question}
Answer:
```

---

## 4. Issue #2 — Retrieval Missed the Right FAQ Chunk

### Ground Truth (in `FAQ.docx`)

| FAQ | Answer |
|-----|--------|
| Q3. What is the cost to register? | **Registration is FREE** |
| Q4. Profile creation | Simple and entirely **free** |
| Payment Q1. Free vs paid? | Free = 3 connections; Paid = many more |

### What Retrieval Actually Returned (top 4, `k=4`)

| Rank | Chunk content | Relevant? |
|------|---------------|----------|
| 1 | Refund/cancellation liability | Partial |
| 2 | Terms & conditions (features) | Partial |
| 3 | FAQ Q7 — registration takes 10–15 min | Wrong aspect |
| 4 | FAQ Q8 — form is lengthy | Wrong aspect |

The pricing FAQ **was in the index** but never reached the LLM. Several factors contributed — see below.

### Why Did Retrieval Miss the Right Chunk?

Retrieval failure is rarely caused by a single setting. In our case, **multiple factors worked together**. A low `k` value was one contributor — not the only one.

```
All chunks ranked by similarity score
────────────────────────────────────────────────────────────
 #1  Refund policy          ← sent to LLM
 #2  Terms & conditions     ← sent to LLM
 #3  FAQ Q7 (reg. time)    ← sent to LLM
 #4  FAQ Q8 (lengthy form)  ← sent to LLM  ← k=4 stops here
 #5  FAQ Q3 (cost = FREE)  ← never reached LLM
 #6  ...
```

---

#### Factor 1 — Low `k` value (`k=4`)

`k` controls how many top-ranked chunks are passed to the LLM.

```python
# app/services/rag_service.py
search_kwargs={"k": 4}
```

If the correct chunk ranks **5th or lower**, it is silently dropped — even when it exists in the vector DB. Here, the pricing FAQ was likely just outside the top 4.

| k | Trade-off |
|---|-----------|
| Low (2–4) | Faster, less noise; **easy to miss** a relevant chunk ranked slightly lower |
| Higher (6–8) | Better recall; more tokens and noise for the LLM |

> `k=4` did not cause wrong chunks to rank high — but it **prevented the right chunk from being included** once ranking was already imperfect.

---

#### Factor 2 — Semantic similarity confusion

The question and the correct FAQ use **different wording**:

| User question | FAQ chunk |
|---------------|-----------|
| *"Is the **platform free**?"* | *"What is the **cost to register**? → Registration is FREE"* |

Embeddings match by meaning, not keywords. Chunks about registration **time**, contacting members (**free**), and liability scored similarly — related topics, wrong answer.

---

#### Factor 3 — Noisy knowledge base

`consolidated_full_corpus.txt` (~32 MB) contained unrelated text with "free" and "registration". Junk chunks could **outrank** the real FAQ before `k` even came into play.

---

#### Factor 4 — Chunking strategy

Settings: `chunk_size=1000`, `chunk_overlap=200`.

FAQ.docx was split into large chunks. Q7 (registration **time**) and Q3 (registration **cost**) landed in **different chunks**. The time chunk matched the query more strongly because both mention "registration".

> **Improvement idea:** one Q&A pair per chunk for FAQ documents.

---

#### Factor 5 — Query vs document embedding mismatch

Some embedders need different prefixes for queries vs documents (e.g. Nomic `search_query:` / `search_document:`). Without them, *"is it free?"* aligns poorly with *"cost to register"* in vector space.

---

#### Factor 6 — Embedding model quality

Weaker embedders conflate "registration **process**" with "registration **price**". Better models (Qwen3, BGE-M3) improved ranking without changing `k`.

---

#### Factor 7 — No hybrid search

Pure vector search struggles when the question says **"free"** but the chunk says **"cost"**. Keyword search (BM25) would help; our pipeline uses dense similarity only.

---

#### Factor 8 — Duplicate chunks wasting slots

Logs showed the **same FAQ section retrieved twice**, using 2 of 4 available `k` slots on near-duplicates.

---

#### Factor 9 — `.docx` extraction flattens structure

`Docx2txtLoader` removes Q&A formatting. One chunk can blend several questions, diluting the vector for any single answer.

---

### Summary — all contributing factors

| # | Factor | How it hurt retrieval |
|---|--------|----------------------|
| 1 | **`k=4` too low** | Correct chunk ranked ~#5; cut off before reaching LLM |
| 2 | **Semantic confusion** | "Free platform" ≠ "cost to register" in embedding space |
| 3 | **Junk corpus** | Noise chunks outranked real FAQ content |
| 4 | **Chunking** | Registration time and cost split across different chunks |
| 5 | **No query prefixes** | Query/document embedding mismatch |
| 6 | **Weak embedder** | Poor ranking of the right chunk (fixed with Qwen3) |
| 7 | **No hybrid search** | "Free" in question didn't boost "cost" chunk |
| 8 | **Duplicate results** | 2 of 4 `k` slots wasted on repeats |
| 9 | **Flat `.docx` text** | FAQ structure lost during extraction |

> **Takeaway:** Low `k` was **one reason** the pricing FAQ never reached the LLM — but only after other factors (noise, chunking, ranking quality) had already pushed it below the top 4. Fixing retrieval means addressing **ranking quality first**, then tuning `k`, and optionally adding re-ranking.

### Optional Retrieval Tweaks

#### What is `k`?

`k` = number of chunks passed to the LLM after vector search.

```python
# app/services/rag_service.py
search_kwargs={"k": 4}  # only top 4 chunks
```

**Flow:** Question → embed → compare to all chunks → take top `k` → send to LLM

| k value | Effect |
|---------|--------|
| 2–4 | Fast, less noise; easy to **miss** the right chunk |
| 6–8 | Better recall; more tokens and noise for the LLM |

For *"is it free?"*, the pricing FAQ might be chunk #5–#6 while #1–#4 are about registration **time** — both mention "registration" but answer different questions.

#### Option A — Increase `k` to 6–8

**Pros:** Simple change; better chance the right chunk is included  
**Cons:** Longer prompt, more RAM; small LLMs may get confused by extra text

#### Option B — Add a Re-ranker

```
Question
   ↓
Retrieve top 20–50 candidates  (high k)
   ↓
Re-ranker scores (question, chunk) pairs
   ↓
Keep top 4–6 for the LLM
```

| Step | Method | Strength |
|------|--------|----------|
| 1. Embed search | Cosine similarity | Fast, approximate |
| 2. Re-rank | BGE-reranker, cross-encoder | Reads Q+A together; better precision |

**Analogy:** Embeddings = keyword search in a library. Re-ranker = librarian reading the shortlist and picking the book that actually answers your question.

> **Note:** After removing junk data and switching to Qwen3 embeddings, retrieval improved significantly. `k=6` is a low-effort next step if some questions still miss the right chunk.

---

## 5. Issue #3 — Noisy Knowledge Base

### Problem

`consolidated_full_corpus.txt` (~32 MB) contained unrelated Odia translation data with thousands of lines mentioning words like **"free"**, **"registration"**, and **"register"** in unrelated contexts.

### Effect

- Polluted vector search
- Pushed out real HR FAQ chunks from top-k results
- Inflated chunk count before cleanup

### Fix

- Removed the file from `data/docs/Text_documents/`
- Rebuilt vector stores → **66 chunks** from 3 clean HR documents:
  - `FAQ.docx`
  - `terms and condition of vedicmarriage@ai.docx`
  - `VedicMarriage_Refund_Cancellation_Policy.docx`

---

## 6. Other Issues Discovered

| Issue | Symptom | Fix |
|-------|---------|-----|
| **Streamlit ≠ API config** | Different answers in UI vs API | Unified defaults via `rag_stacks.py` |
| **Missing `docx2txt`** | Vector build failed on `.docx` | `pip install docx2txt` |
| **Ollama too old (0.6.8)** | Couldn't pull `qwen3-embedding` | Upgraded to 0.31.1 via Homebrew |
| **7B models on 8 GB Mac** | System hung / swapped | Use ~1 GB models (Qwen, Llama 3.2 1B) |
| **Random LLM + embedding combos** | Missing vector store errors | Curated stacks in `rag_stacks.py` |
| **Missing vector indexes** | Init failed for some stacks | Built 4 stores: `ollama_qwen3-0.6b`, `ollama_bge-m3`, `ollama_nomic`, `hf_bge-base` |

---

## 7. How We Diagnosed Each Issue

| Technique | What it revealed |
|-----------|------------------|
| **Streamlit / LangChain verbose logs** | Full prompt + retrieved context before LLM answer |
| **Grep / unzip `FAQ.docx`** | Ground truth answer exists in documents |
| **Compare output to prompt template** | Prompt parroting vs generation failure |
| **`ollama list` / `ollama --version`** | Model availability and version limits |
| **Inspect `data/chroma_db/`** | Which vector indexes exist vs what stacks need |
| **RAM + model size estimate** | Why 7B models hang on 8 GB Mac |
| **Trace config in `routes.py` vs `streamlit_app.py`** | Two different RAG pipelines |

---

## 8. Final Architecture

### Default RAG Stack

| Component | Choice |
|-----------|--------|
| LLM | `qwen2.5:1.5b` (Ollama) |
| Embeddings | `qwen3-embedding:0.6b` (Ollama) |
| Vector DB | `data/chroma_db/ollama_qwen3-0.6b/` |
| Chunks | 66 |

### Available Stacks (9 total)

Defined in `app/core/rag_stacks.py` — only validated LLM + embedding combinations are exposed in Streamlit.

### Key Lessons

1. **RAG fails in layers** — check retrieval, prompt, and generation separately.
2. **Good retrieval ≠ good answer** — a weak LLM ignores correct context.
3. **One embedder = one vector index** — rebuild Chroma when changing embeddings.
4. **Corpus hygiene matters** — irrelevant docs poison search.
5. **Match models to hardware** — 8 GB Mac → ~1 GB LLMs; Ollama for both LLM + embeddings saves RAM.
6. **Curate stacks** — don't expose every random LLM × embedding combination.

### Verification Question

> *"Is the vedic marriage platform free?"*

Expected: Yes — registration is free; free tier has limited features; Premium adds more.

---

## 9. Case Study — Subscription Plans (Retrieval vs Synthesis)

### Question

> **What are different subscription plans the site has?**

### Ground truth (split across FAQ chunks)

| FAQ | Content |
|-----|---------|
| Q1 | Free member → up to **3 connections** |
| Q2 | **Premium**, **Premium Plus**, **Vedic Recommender** (with durations/limits) |

### What went wrong (first)

- Answer listed only 3 **premium** tiers — **Free** missing.
- Sources: `FAQ.docx` + `terms and condition...docx` (2 files, 3 chunks).

### Key finding #1 — Raising `k` alone does not fix bad ranking

| k | Result |
|---|--------|
| k=4, k=6 | Wrong — payment FAQ was rank **~#15** |
| k≥15 | Might include it — but noisy, slow, worse for 1.5B LLM |

> **Takeaway:** Fix **chunking + ranking** first. `k` is a margin, not a substitute.

### Fixes that worked

| Fix | Why |
|-----|-----|
| **FAQ Q&A chunking** (`document_service.py`) | One chunk per `Q1.`, `Q2.` — plans chunk ranks #1 |
| **Wipe Chroma before rebuild** | Old + new chunks stacked (159 vs 93) — polluted search |
| **Cross-encoder rerank** | Fetch 15–20 → rerank → top 3–5 for LLM |
| **Exclude `~$*.docx`** | Word lock file broke rebuild (`BadZipFile`) |

---

## 10. Reranking Pipeline (General — No Prompt Hacks)

```
Question
   ↓
[1] Vector search — fetch 15–20 candidates (bi-encoder, fast)
   ↓
[2] Cross-encoder rerank — score (question, chunk) pairs
   ↓
[3] Length penalty — down-rank long legal prose
   ↓
[4] Diversity — skip near-duplicate chunks; max 2 per source file
   ↓
[5] Order — shortest chunks first in context (small LLMs read the start best)
   ↓
Top k chunks → LLM
```

**Model:** `cross-encoder/ms-marco-MiniLM-L-6-v2` (~80 MB)  
**Config:** `app/core/config.py` — `RERANK_CANDIDATES`, `TOP_K_RESULTS`, `RERANK_ENABLED`

| Approach | Verdict |
|----------|---------|
| Hardcoded plan/prompt rules | Fragile — avoid |
| Rerank + better chunking | Scales to new questions |
| Bigger LLM (`gemma2:2b`) | Better multi-chunk synthesis; more RAM |

---

## 11. Why the LLM Misses Free (Even When Context Has It)

Retrieval often **succeeds** — both FAQ chunks reach the LLM:

```
Chunk 1: Q1 — "free member … 3 members"
Chunk 2: Q2 — "premium memberships … Premium / Plus / Recommender"
Chunk 3: terms — "access to the website is free … subscription plan"
```

**`qwen2.5:1.5b` still answers only 3 premium tiers** because:

1. **RAG does not learn** — each turn is retrieve → generate once; no memory across chunks.
2. **Inconsistent naming** — Q1 says "free member", Q2 says "premium memberships"; model latches onto Q2's framing.
3. **Small model limit** — weak at merging facts from separate chunks into one list.
4. **Terms chunk noise** — word **"site"** in the query pulls legal doc; reinforces "premium" wording.

### Best fix (data, not prompt)

Merge at **index time** into one overview chunk:

```
Subscription plans: Free (3 connections) | Premium | Premium Plus | Vedic Recommender
```

One retrieval hit → one clear context → even 1.5B can answer.

---

## 12. Same Topic, Different Answers — Why?

Observed: two similar questions gave different completeness.

| Query | Free tier? | Extra chunk |
|-------|------------|-------------|
| `…plan the **site** has?` | Often missing | **terms** (matches "site") |
| `…plan the ?` | Sometimes all 4 | **refund policy** lists "Free, Premium, Plus, Recommender" in one sentence |

**Causes:**

| # | Cause |
|---|--------|
| 1 | **Slight retrieval change** — refund chunk has all plan names in one line (easy to copy) |
| 2 | **LLM randomness** — `temperature=0.3` still varies |
| 3 | **Chat history** — Streamlit follow-ups rephrase the question via `ConversationalRetrievalChain` |

> Same chunks + different 3rd chunk + 1.5B model = inconsistent list answers.

---

## 13. Why Two Sources Appear

Retriever returns **k chunks**; UI **dedupes by filename**.

Example (k=3):

| Chunk | File |
|-------|------|
| 1 | FAQ.docx (Q1 free) |
| 2 | FAQ.docx (Q2 plans) |
| 3 | terms.docx |

**Sources shown:** FAQ.docx + terms.docx → **2 lines** (not 3).

Terms appears because the query contains **"site"** — matches legal language, not because it's the best answer source.

---

## 14. How to Inspect Retrieved Chunks

### Quick CLI (from project root)

```bash
conda activate llm
python -c "
from app.services.rag_service import RAGService
from app.core.rag_stacks import get_default_stack
from app.services.embedding_service import EmbeddingType

q = 'YOUR QUESTION HERE'
stack = get_default_stack()
rag = RAGService(model_name=stack['llm'],
    embedding_type=EmbeddingType(stack['embedding_type']),
    embedding_model=stack['embedding_model'])

print('=== VECTOR SEARCH (candidates) ===')
for i, d in enumerate(rag.vectorstore.similarity_search(q, k=10), 1):
    print(i, d.metadata['source'].split('/')[-1], len(d.page_content))

print('=== FINAL (after rerank) ===')
for i, d in enumerate(rag.retriever._get_relevant_documents(q), 1):
    print('---', i, '---')
    print(d.page_content)
"
```

### Other methods

| Method | Where |
|--------|--------|
| Streamlit terminal | `verbose=True` → full `Context:` block in logs |
| `response['source_documents']` | One entry per chunk (before filename dedupe) |
| Compare stages | `similarity_search(k=20)` vs `retriever._get_relevant_documents(q)` |

---

## 15. Updated Lessons (Checklist)

| # | Lesson |
|---|--------|
| 1 | **RAG fails in layers** — retrieval, ranking, prompt, generation are independent |
| 2 | **Ranking before k** — wrong chunk at #15 is not fixed by k=6 |
| 3 | **Rebuild = wipe Chroma dir** — appending leaves stale chunks |
| 4 | **FAQ → one Q&A per chunk** — list answers need focused chunks |
| 5 | **Rerank > raw top-k** — cross-encoder + diversity beats blind k increase |
| 6 | **Small LLM ≠ synthesizer** — 1.5B struggles to merge split facts across chunks |
| 7 | **Corpus hygiene** — junk files and Word lock files (`~$*.docx`) break builds |
| 8 | **Query wording shifts retrieval** — "site" → terms; accidental refund chunk → fuller answer |
| 9 | **Sources = unique files** — multiple FAQ chunks still show one FAQ source |
| 10 | **Inspect before blaming the LLM** — check Stage 1 vs Stage 2 retrieval |

### Verification questions

| Question | What to check |
|----------|----------------|
| *Is the platform free?* | FAQ pricing chunk in top-k |
| *What subscription plans?* | Free + 3 paid tiers; prefer single overview chunk |
| *Refund policy?* | Refund doc ranks above unrelated FAQ |